In [1]:
import os
import time
import shutil
import requests
from dotenv import load_dotenv
from tqdm import tqdm
from bs4 import BeautifulSoup
import pandas as pd

load_dotenv()

USER_AGENT = os.getenv("SEC_USER_AGENT")

if not USER_AGENT:
    raise ValueError("SEC_USER_AGENT not found in .env")

headers = {
    "User-Agent": USER_AGENT
}

tickers = [
    "AAPL", "ABBV", "ABT", "ACN", "ADBE", "AIG", "AMD", "AMGN", "AMT", "AMZN", 
    "AVGO", "AXP", "BA", "BAC", "BK", "BKNG", "BLK", "BMY", "C", "CAT", 
    "CHTR", "CL", "CMCSA", "COF", "COP", "COST", "CRM", "CSCO", "CVS", "CVX", 
    "DHR", "DIS", "DOW", "DUK", "EMR", "EXC", "F", "FDX", "GD", "GE", 
    "GILD", "GM", "GOOG", "GOOGL", "GS", "HD", "HON", "IBM", "INTC", "JNJ", 
    "JPM", "KHC", "KO", "LLY", "LMT", "LOW", "MA", "MCD", "MDLZ", "MDT", 
    "MET", "META", "MMM", "MO", "MRK", "MS", "MSFT", "NEE", "NFLX", "NKE", 
    "NVDA", "ORCL", "PEP", "PFE", "PG", "PLTR", "QCOM", "RTX", "SBUX", "SCHW", 
    "SO", "SPG", "T", "TMO", "TMUS", "TSLA", "TXN", "UBER", "UNH", "UNP", 
    "UPS", "USB", "V", "VZ", "WFC", "WMT", "XOM", "AEP", "BIIB"
]

start_year = 2014
end_year = 2026

shutil.rmtree("data/10K", ignore_errors=True)
shutil.rmtree("data/10Q", ignore_errors=True)

os.makedirs("data/10K", exist_ok=True)
os.makedirs("data/10Q", exist_ok=True)

ticker_url = "https://www.sec.gov/files/company_tickers.json"
ticker_data = requests.get(ticker_url, headers=headers).json()

ticker_to_cik = {}

for item in ticker_data.values():
    ticker_to_cik[item["ticker"]] = str(item["cik_str"]).zfill(10)

def download_filing(ticker, form_type, filing_date, accession_number, primary_doc):
    cik = ticker_to_cik[ticker]
    accession_clean = accession_number.replace("-", "")

    doc_url = (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{int(cik)}/{accession_clean}/{primary_doc}"
    )

    if form_type == "10-K":
        folder = "data/10K"
    elif form_type == "10-Q":
        folder = "data/10Q"
    else:
        return False

    file_name = f"{ticker}_{form_type}_{filing_date}_{accession_number}.html"
    file_name = file_name.replace("/", "-").replace(":", "-")
    file_path = os.path.join(folder, file_name)

    if os.path.exists(file_path):
        return False

    response = requests.get(doc_url, headers=headers, timeout=30)

    if response.status_code == 200:
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(response.text)
        return True

    return False


def process_filing_batch(ticker, filing_batch):
    forms = filing_batch["form"]
    accession_numbers = filing_batch["accessionNumber"]
    filing_dates = filing_batch["filingDate"]
    primary_docs = filing_batch["primaryDocument"]

    downloaded_count = 0

    for i in range(len(forms)):
        form_type = forms[i]
        filing_date = filing_dates[i]

        if form_type not in ["10-K", "10-Q"]:
            continue

        filing_year = int(filing_date[:4])

        if filing_year < start_year or filing_year > end_year:
            continue

        try:
            downloaded = download_filing(
                ticker,
                form_type,
                filing_date,
                accession_numbers[i],
                primary_docs[i],
            )

            if downloaded:
                downloaded_count += 1

            time.sleep(0.15)

        except Exception as e:
            print(f"Skipped filing: {ticker} {form_type} {filing_date} {e}")

    return downloaded_count


def download_company_filings(ticker):
    if ticker not in ticker_to_cik:
        print(f"Skipped {ticker}: CIK not found")
        return

    cik = ticker_to_cik[ticker]
    submissions_url = f"https://data.sec.gov/submissions/CIK{cik}.json"

    response = requests.get(submissions_url, headers=headers, timeout=30)

    if response.status_code != 200:
        print(f"Skipped {ticker}: cannot access SEC submissions")
        return

    data = response.json()

    downloaded_count = 0

    recent = data["filings"]["recent"]
    downloaded_count += process_filing_batch(ticker, recent)

    archive_files = data["filings"].get("files", [])

    for archive in archive_files:
        archive_name = archive["name"]
        archive_url = f"https://data.sec.gov/submissions/{archive_name}"

        try:
            archive_response = requests.get(archive_url, headers=headers, timeout=30)

            if archive_response.status_code != 200:
                continue

            archive_data = archive_response.json()
            downloaded_count += process_filing_batch(ticker, archive_data)

            time.sleep(0.15)

        except Exception as e:
            print(f"Skipped archive: {ticker} {archive_name} {e}")

    print(f"{ticker}: downloaded {downloaded_count} filings")


for ticker in tqdm(tickers):
    download_company_filings(ticker)
    time.sleep(0.3)

print("Download complete")

/Users/ruizisun/Desktop/IBM/financial risk agent coding/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
  0%|          | 0/99 [00:00<?, ?it/s]

AAPL: downloaded 49 filings


  1%|          | 1/99 [00:13<22:31, 13.79s/it]

ABBV: downloaded 49 filings


  2%|▏         | 2/99 [00:27<22:25, 13.87s/it]

...

AEP: downloaded 49 filings


 99%|█████████▉| 98/99 [25:50<00:15, 15.82s/it]

BIIB: downloaded 49 filings


100%|██████████| 99/99 [26:05<00:00, 15.81s/it]

Download complete


In [15]:
all_files = os.listdir("data/10K") + os.listdir("data/10Q")

years = sorted(
    set(
        int(file.split("_")[2][:4])
        for file in all_files
        if len(file.split("_")) > 2 and file.split("_")[2][:4].isdigit()
    )
)

print(years)

[2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [16]:
def extract_text(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "ix:nonfraction", "ix:nonnumeric"]):
        tag.extract()

    text = soup.get_text(separator=" ")
    text = " ".join(text.split())

    return text

In [17]:
def get_positions(lower_text):
    positions = []
    start = 0

    while True:
        position = lower_text.find("risk factors", start)
        if position == -1:
            break
        positions.append(position)
        start = position + 1

    return positions


def extract_section(text, positions, idx=1):
    if len(positions) == 0:
        return text[:50000]

    if idx >= len(positions):
        idx = 0

    start = positions[idx]
    end = positions[idx + 1] if idx + 1 < len(positions) else start + 50000

    return text[start:end]


def analyze_risk_text(risk_factors_text):
    keywords = {
        "regulation": ["regulation", "regulatory", "compliance", "legal", "sanctions"],
        "financial": ["liquidity", "debt", "cash flow", "revenue", "costs"],
        "market": ["competition", "demand", "decline", "market"],
        "operations": ["supply chain", "disruption", "operations", "shortage"],
        "technology": ["cybersecurity", "data breach", "system failure", "technology"]
    }

    risk_summary = {}
    lower_text = risk_factors_text.lower()

    for category, words in keywords.items():
        count = sum(lower_text.count(word) for word in words)
        risk_summary[category] = count

    return risk_summary


def build_risk_summary(file_path):
    clean_text = extract_text(file_path)
    lower_text = clean_text.lower()
    positions = get_positions(lower_text)

    risk_factors_text = extract_section(clean_text, positions, idx=1)
    risk_summary = analyze_risk_text(risk_factors_text)

    return risk_summary, risk_factors_text

In [18]:
def get_company_files_by_type(company, filing_type):
    folder = f"data/{filing_type.replace('-', '')}"
    files = os.listdir(folder)
    company_files = [f for f in files if f.startswith(company + f"_{filing_type}")]
    company_files.sort()
    return company_files


def get_year_from_file(file_name):
    parts = file_name.split("_")
    for part in parts:
        if part.startswith("20"):
            return int(part[:4])
    return None


def get_all_companies_by_type(filing_type):
    folder = f"data/{filing_type.replace('-', '')}"
    files = os.listdir(folder)
    companies = sorted(list(set([f.split(f"_{filing_type}")[0] for f in files if f"_{filing_type}" in f])))
    return companies

In [19]:
def build_filing_summary_cache(filing_type):
    folder = f"data/{filing_type.replace('-', '')}"
    files = os.listdir(folder)

    summary_cache = {}

    for file in tqdm(files, desc=f"Parsing {filing_type} files"):
        file_path = os.path.join(folder, file)

        try:
            risk_summary, risk_text = build_risk_summary(file_path)
            summary_cache[file] = risk_summary

        except Exception as e:
            print("Skipped parsing:", filing_type, file, e)

    return summary_cache


def build_portfolio_results_for_type(filing_type):
    portfolio_results = []
    companies = get_all_companies_by_type(filing_type)
    folder = f"data/{filing_type.replace('-', '')}"

    summary_cache = build_filing_summary_cache(filing_type)

    for company in tqdm(companies, desc=f"Building {filing_type} comparisons"):
        files = get_company_files_by_type(company, filing_type)

        year_files = []
        for file in files:
            year = get_year_from_file(file)
            if year is not None and file in summary_cache:
                year_files.append((year, file))

        year_files.sort()

        for i in range(len(year_files) - 1):
            older_year, older_file = year_files[i]
            newer_year, newer_file = year_files[i + 1]

            older_summary = summary_cache[older_file]
            newer_summary = summary_cache[newer_file]

            risk_change = {}
            for key in newer_summary:
                risk_change[key] = newer_summary[key] - older_summary[key]

            top_increase = max(risk_change, key=risk_change.get)
            top_decrease = min(risk_change, key=risk_change.get)

            portfolio_results.append({
                "company": company,
                "filing_type": filing_type,
                "older_year": older_year,
                "newer_year": newer_year,
                "older_file": older_file,
                "newer_file": newer_file,
                "top_increase": top_increase,
                "top_decrease": top_decrease,
                "regulation": risk_change["regulation"],
                "financial": risk_change["financial"],
                "market": risk_change["market"],
                "operations": risk_change["operations"],
                "technology": risk_change["technology"]
            })

    return portfolio_results

In [20]:
portfolio_results_10k = build_portfolio_results_for_type("10-K")
portfolio_results_10q = build_portfolio_results_for_type("10-Q")

df_10k = pd.DataFrame(portfolio_results_10k)
df_10q = pd.DataFrame(portfolio_results_10q)

df_10k.to_csv("portfolio_10k.csv", index=False)
df_10q.to_csv("portfolio_10q.csv", index=False)

print("10K rows:", len(df_10k))
print("10Q rows:", len(df_10q))

Building 10-Q comparisons: 100%|██████████| 99/99 [00:00<00:00, 700.89it/s]

10K rows: 1126
10Q rows: 3376


In [25]:
print("10K rows:", len(df_10k))
print("10Q rows:", len(df_10q))

print("10K companies:", df_10k["company"].nunique())
print("10Q companies:", df_10q["company"].nunique())

print("10K files:", len(os.listdir("data/10K")))
print("10Q files:", len(os.listdir("data/10Q")))

10K rows: 1126
10Q rows: 3376
10K companies: 99
10Q companies: 99
10K files: 1225
10Q files: 3475


In [26]:
# After changing app.py then paste "./.venv/bin/python -m streamlit run app.py" in terminal to launch the Streamlit app. Then you can view the portfolio comparison dashboard.

In [27]:
# Update Instructions
# GitHub: in VSCode terminal：
# git status
# git add .
# git commit -m "update"
# git pull origin main --rebase
# git push

# Streamlit: Click Reboot app
# https://share.streamlit.io/

In [28]:
# 🔗 Live App (Streamlit):
# https://lumen-financial-risk-agent.streamlit.app/

# 🔗 GitHub Repository:
# https://github.com/autosoon-commits/Lumen-Financial-Risk-Agent